In [ ]:
from fastai.vision.all import *
from torch.nn import *


class Dat:
    a = torch.from_numpy(np.memmap(f"../tmp/news2016zhMin100.H", "h"))
    m = np.array([open("../tmp/news2016zhMin100.txt").read()]).view("U1")

    def __init__(s, w, b, n=1):
        s.a = s.a.as_strided([len(s.a) - w + 1, w], [1, 1])
        s.b = b
        s.n = n

    def __len__(s):
        return len(s.a) // (s.a.size(1) * s.b * s.n)

    def __iter__(s):
        for x in range(len(s)):
            x = s.a[torch.randint(len(s.a), [s.b])].to("cuda", torch.long)
            yield x, x


class Transpose(Module):
    @staticmethod
    def forward(x):
        return x.transpose(1, 2)


class ConvBn(Sequential):
    def __init__(_, i, o, w=3, s=1, act=1):
        super().__init__(Conv1d(i, o, w, s, w // 2, bias=0), BatchNorm1d(o, track_running_stats=0), *[ReLU(1)] if act else [])


class ConvTransBn(Sequential):
    def __init__(_, i, o, w=3, s=1, act=1):
        super().__init__(
            ConvTranspose1d(i, o, w, s, w // 2, s // 2, bias=0), BatchNorm1d(o, track_running_stats=0), *[ReLU(1)] if act else []
        )


class Res(Sequential):
    def forward(_, x):
        return super().forward(x) + x


class Cnn(Sequential):
    def __init__(s, *a, n=0):
        super().__init__(*a)
        s.n = n

    def forward(s, x):
        for _ in range(s.n or int(math.log2(x.size(2)))):
            x = super().forward(x)
        return x


class PyramidAutoEncoder(Sequential):
    def forward(s, x):
        y = [x]
        while x.size(2) > 1:
            x = s[0](x)
            y.append(x)
        y = list(s[1](torch.cat(y, 2)).split([y.size(2) for y in y], 2))
        x = y.pop()
        while y:
            x = torch.cat((y.pop(), s[2](x)))
        return x


class PyramidEncoder(Sequential):
    def forward(s, x):
        y = [x]
        for s in s:
            x = s(x)
            y.append(x)
        return torch.cat(y, 2)


class PyramidDecoder(Sequential):
    def forward(s, x):
        x = list(x.split([2 ** i for i in range(int(math.log2(x.size(2))) + 1)[::-1]], 2))
        y = x.pop()
        for s in s:
            y = torch.cat((x.pop(), s(y)))
        return y


class PyramidCrossEntropyLoss(CrossEntropyLoss):
    def forward(s, x, y):
        x = x.view(-1, len(y), *x.shape[1:])
        return super().forward(x.transpose(1, 2), y.broadcast_to(len(x), *y.shape))


def PyramidAccuracy(n):
    F = []
    for n in range(n):

        def f(x, y, n=n):
            with torch.no_grad():
                return accuracy(x.view(-1, len(y), *x.shape[1:])[n], y, 1)

        f.__name__ = str(n)
        F.append(f)
    return F

In [ ]:
m = Learner(
    DataLoaders(Dat(256, 2 ** 6, 100), Dat(256, 2 ** 6, 50000)),
    Sequential(
        Embedding(684, 256),
        Transpose(),
        Sequential(Res(ConvBn(256), ConvBn(256)), AvgPool1d(2)),
        Sequential(Res(ConvBn(256), ConvBn(256)), AvgPool1d(2)),
        Sequential(Res(ConvBn(256), ConvBn(256)), AvgPool1d(2)),
        Sequential(Res(ConvBn(256), ConvBn(256)), AvgPool1d(2)),
        Sequential(Res(ConvBn(256), ConvBn(256)), AvgPool1d(2)),
        Sequential(Res(ConvBn(256), ConvBn(256)), AvgPool1d(2)),
        Sequential(Res(ConvBn(256), ConvBn(256)), AvgPool1d(2)),
        Sequential(Res(ConvBn(256), ConvBn(256)), AvgPool1d(2)),
        Sequential(Res(ConvTransBn(256), ConvTransBn(256)), ConvTranspose1d(256, 256, 3, 2, 1, 1, bias=0)),
        Sequential(Res(ConvTransBn(256), ConvTransBn(256)), ConvTranspose1d(256, 256, 3, 2, 1, 1, bias=0)),
        Sequential(Res(ConvTransBn(256), ConvTransBn(256)), ConvTranspose1d(256, 256, 3, 2, 1, 1, bias=0)),
        Sequential(Res(ConvTransBn(256), ConvTransBn(256)), ConvTranspose1d(256, 256, 3, 2, 1, 1, bias=0)),
        Sequential(Res(ConvTransBn(256), ConvTransBn(256)), ConvTranspose1d(256, 256, 3, 2, 1, 1, bias=0)),
        Sequential(Res(ConvTransBn(256), ConvTransBn(256)), ConvTranspose1d(256, 256, 3, 2, 1, 1, bias=0)),
        Sequential(Res(ConvTransBn(256), ConvTransBn(256)), ConvTranspose1d(256, 256, 3, 2, 1, 1, bias=0)),
        Sequential(Res(ConvTransBn(256), ConvTransBn(256)), ConvTranspose1d(256, 256, 3, 2, 1, 1, bias=0)),
        Conv1d(256, 684, 1),
    ).cuda(),
    CrossEntropyLoss(),
    cbs=[
        ShowGraphCallback,
        SaveModelCallback(fname="model5-autoencoder-Copy15", with_opt=1, reset_on_fit=0),
    ],
    metrics=partial(accuracy, axis=1),
    wd=0.01,
)
m.lr_find()

In [ ]:
class Add(Sequential):
    def forward(s, x):
        return s[0](x) + s[1](x)


class Conv(Sequential):
    def __init__(_, i, o=0, w=3, s=1, act=1):
        o = o or i
        super().__init__(Conv1d(i, o, w, s, w // 2, bias=0), BatchNorm1d(o, track_running_stats=0), *[ReLU(1)] if act else [])


class Res(Sequential):
    def __init__(_, i, o=0, s=1):
        o = o or i
        m = o // 4
        super().__init__(
            Add(
                Seq(Conv(i, m, 1), Conv(m, s=s), Conv(m, o, 1, act=0)),
                Seq(*[AvgPool1d(s)] if s > 1 else [], *[Conv(i, o, 1, act=0)] if i != o else []),
            ),
            ReLU(1),
        )


class ResBlock(Sequential):
    def __init__(_, i, s=2):
        super().__init__(Res(i, i * 2, s), Res(i * 2, i * 2))


class ConvTrans(Sequential):
    def __init__(_, i, o=0, w=3, s=1, act=1):
        o = o or i
        super().__init__(
            ConvTranspose1d(i, o, w, s, w // 2, s // 2, bias=0), BatchNorm1d(o, track_running_stats=0), *[ReLU(1)] if act else []
        )


class ResTrans(Sequential):
    def __init__(_, i, o=0, s=1):
        o = o or i
        m = o // 4
        super().__init__(
            Add(
                Seq(ConvTrans(i, m, 1), Conv(m, s=s), Conv(m, o, 1, act=0)),
                Seq(*[AvgPool1d(s)] if s > 1 else [], *[Conv(i, o, 1, act=0)] if i != o else []),
            ),
            ReLU(1),
        )

In [ ]:
Seq(
    Embedding(684, 8),
    Transpose(),
    Conv(8, s=2),
    Conv(8),
    Conv(8, 16),
    MaxPool1d(3, 2, 1),
    ResBlock(16, 1),
    ResBlock(32),
    ResBlock(64),
    ResBlock(128),
    AdaptiveAvgPool1d(1),
    ResBlockTrans(256),
    ResBlockTrans(128),
    ResBlockTrans(64),
    ResBlockTrans(32, 1),
    ConvTranspose1d(16, 16, 3, 2, 1, 1, bias=0),
    ConvTrans(16, 8),
    ConvTrans(8),
    ConvTrans(8, s=2),
    Conv1d(8, 684, 1),
)

In [ ]:
np.log(684)

In [ ]:
ConvTranspose1d(2,4,1)(torch.rand(3,2,5))

In [ ]:
ResNetShorcutD.forward??

In [ ]:
a = AutoModel.from_name('resnet26d')

In [ ]:
a.head

In [ ]:
a.encoder.stem[0].forward

In [ ]:
ResNet.ResNet

In [ ]:
AutoModel.from_pretrained('../resnet26d')

In [ ]:
from pickle import load
a = torch.load('../resnet26d/archive/data.pkl', 'rb')

In [ ]:
ResNet

In [ ]:
m.fit_one_cycle(100, .001, 9, 99, .01)